In [6]:
from pymavlink import mavutil


# =========================
# Change this path only
# =========================
TLOG_PATH = "two_land_and_go_to_waypoint.tlog"



def cmd_name(cmd_id):
    try:
        e = mavutil.mavlink.enums["MAV_CMD"].get(int(cmd_id))
        return e.name if e else f"UNKNOWN_MAV_CMD_{cmd_id}"
    except Exception:
        return f"UNKNOWN_MAV_CMD_{cmd_id}"


def frame_name(frame_id):
    try:
        e = mavutil.mavlink.enums["MAV_FRAME"].get(int(frame_id))
        return e.name if e else f"UNKNOWN_MAV_FRAME_{frame_id}"
    except Exception:
        return f"UNKNOWN_MAV_FRAME_{frame_id}"


def print_mission_item(msg):
    msg_type = msg.get_type()

    seq = int(getattr(msg, "seq", -1))
    frame = int(getattr(msg, "frame", -1))
    command = int(getattr(msg, "command", -1))

    p1 = float(getattr(msg, "param1", 0.0))
    p2 = float(getattr(msg, "param2", 0.0))
    p3 = float(getattr(msg, "param3", 0.0))
    p4 = float(getattr(msg, "param4", 0.0))

    x = getattr(msg, "x", None)
    y = getattr(msg, "y", None)
    z = getattr(msg, "z", None)

    if msg_type == "MISSION_ITEM_INT":
        lat = float(x) / 1e7 if x is not None else None
        lon = float(y) / 1e7 if y is not None else None
        alt = float(z) if z is not None else None
    else:
        lat = float(x) if x is not None else None
        lon = float(y) if y is not None else None
        alt = float(z) if z is not None else None

    print("=" * 70)
    print(f"Message Type : {msg_type}")
    print(f"Seq          : {seq}")
    print(f"Command      : {command} ({cmd_name(command)})")
    print(f"Frame        : {frame} ({frame_name(frame)})")
    print(f"Current      : {getattr(msg, 'current', None)}")
    print(f"Autocontinue : {getattr(msg, 'autocontinue', None)}")
    print(f"Param1       : {p1}")
    print(f"Param2       : {p2}")
    print(f"Param3       : {p3}")
    print(f"Param4       : {p4}")
    print(f"Latitude     : {lat}")
    print(f"Longitude    : {lon}")
    print(f"Altitude     : {alt}")


def extract_and_print_mission_items(tlog_path):
    mav = mavutil.mavlink_connection(tlog_path, robust_parsing=True)

    mission_count = None
    item_count = 0

    while True:
        msg = mav.recv_match(blocking=False)

        if msg is None:
            break

        msg_type = msg.get_type()

        if msg_type == "MISSION_COUNT":
            mission_count = int(getattr(msg, "count", -1))
            print("\n" + "#" * 70)
            print(f"MISSION_COUNT found: {mission_count}")
            print("#" * 70)

        elif msg_type in ("MISSION_ITEM", "MISSION_ITEM_INT"):
            item_count += 1
            print_mission_item(msg)

    print("\n" + "#" * 70)
    print(f"Done. Total mission items found: {item_count}")

    if mission_count is not None:
        print(f"Last reported mission count: {mission_count}")

    print("#" * 70)

extract_and_print_mission_items(TLOG_PATH)


######################################################################
MISSION_COUNT found: 7
######################################################################
Message Type : MISSION_ITEM_INT
Seq          : 0
Command      : 22 (MAV_CMD_NAV_TAKEOFF)
Frame        : 6 (MAV_FRAME_GLOBAL_RELATIVE_ALT_INT)
Current      : 1
Autocontinue : 1
Param1       : 0.0
Param2       : 0.0
Param3       : 0.0
Param4       : nan
Latitude     : 47.3979709
Longitude    : 8.5461641
Altitude     : 49.98720169067383
Message Type : MISSION_ITEM_INT
Seq          : 1
Command      : 16 (MAV_CMD_NAV_WAYPOINT)
Frame        : 6 (MAV_FRAME_GLOBAL_RELATIVE_ALT_INT)
Current      : 0
Autocontinue : 1
Param1       : 0.0
Param2       : 0.0
Param3       : 0.0
Param4       : nan
Latitude     : 47.4007566
Longitude    : 8.5398522
Altitude     : 49.98720169067383
Message Type : MISSION_ITEM_INT
Seq          : 2
Command      : 16 (MAV_CMD_NAV_WAYPOINT)
Frame        : 6 (MAV_FRAME_GLOBAL_RELATIVE_ALT_INT)
Current      : 0
A

In [1]:
from datetime import datetime, timezone

from pymavlink import mavutil


# =========================
# Change this path only
# =========================
TLOG_PATH = "two_land_and_go_to_waypoint.tlog"


# False: show only flight/action commands listed below.
# True: show every COMMAND_LONG and COMMAND_INT.
PRINT_ALL_COMMANDS = False


IMPORTANT_COMMAND_NAMES = [
    "MAV_CMD_COMPONENT_ARM_DISARM",
    "MAV_CMD_NAV_TAKEOFF",
    "MAV_CMD_NAV_LAND",
    "MAV_CMD_NAV_RETURN_TO_LAUNCH",
    "MAV_CMD_NAV_LOITER_UNLIM",
    "MAV_CMD_MISSION_START",
    "MAV_CMD_DO_SET_MODE",
    "MAV_CMD_DO_CHANGE_SPEED",
    "MAV_CMD_DO_REPOSITION",
    "MAV_CMD_DO_PAUSE_CONTINUE",
    "MAV_CMD_DO_ORBIT",
    "MAV_CMD_CONDITION_YAW",
    "MAV_CMD_DO_SET_ROI",
    "MAV_CMD_DO_SET_ROI_LOCATION",
    "MAV_CMD_DO_SET_MISSION_CURRENT",
]


IMPORTANT_COMMANDS = {
    getattr(mavutil.mavlink, name)
    for name in IMPORTANT_COMMAND_NAMES
    if hasattr(mavutil.mavlink, name)
}


def enum_name(enum_group, value):
    """Return an enum name, or a readable unknown value."""
    try:
        value = int(value)
        entry = mavutil.mavlink.enums[enum_group].get(value)
        return entry.name if entry else f"UNKNOWN_{enum_group}_{value}"
    except Exception:
        return f"UNKNOWN_{enum_group}_{value}"


def command_name(command):
    return enum_name("MAV_CMD", command)


def frame_name(frame):
    return enum_name("MAV_FRAME", frame)


def message_time(msg):
    """Return the tlog timestamp as UTC when available."""
    timestamp = getattr(msg, "_timestamp", None)

    if timestamp is None:
        return "unknown"

    try:
        return datetime.fromtimestamp(
            float(timestamp),
            tz=timezone.utc,
        ).isoformat()
    except (TypeError, ValueError, OSError):
        return str(timestamp)


def source_string(msg):
    try:
        return f"{msg.get_srcSystem()}:{msg.get_srcComponent()}"
    except Exception:
        return "unknown"


def decode_mode_flags(base_mode):
    """Decode combined MAV_MODE_FLAG bits such as 129."""
    try:
        base_mode = int(base_mode)
        names = []

        enum_values = mavutil.mavlink.enums.get("MAV_MODE_FLAG", {})

        for value, entry in enum_values.items():
            # Only process individual bit values: 1, 2, 4, 8, ...
            if value > 0 and (value & (value - 1)) == 0:
                if base_mode & value:
                    names.append(entry.name)

        return names or ["none"]

    except Exception:
        return ["unknown"]


def decode_px4_custom_mode(custom_mode):
    """
    Decode packed PX4 custom_mode from SET_MODE or HEARTBEAT.

    Bits:
        main mode = bits 16-23
        sub mode  = bits 24-31
    """
    try:
        custom_mode = int(custom_mode)

        main_mode = (custom_mode >> 16) & 0xFF
        sub_mode = (custom_mode >> 24) & 0xFF

        main_name = enum_name("PX4_CUSTOM_MAIN_MODE", main_mode)

        # PX4 main mode 4 is AUTO. Decode the AUTO submode when possible.
        auto_mode = getattr(
            mavutil.mavlink,
            "PX4_CUSTOM_MAIN_MODE_AUTO",
            4,
        )

        if main_mode == auto_mode:
            sub_name = enum_name(
                "PX4_CUSTOM_SUB_MODE_AUTO",
                sub_mode,
            )
        else:
            sub_name = str(sub_mode)

        return {
            "raw": custom_mode,
            "hex": f"0x{custom_mode:08X}",
            "main_mode": main_mode,
            "main_name": main_name,
            "sub_mode": sub_mode,
            "sub_name": sub_name,
        }

    except Exception:
        return None


def print_header(title, msg):
    print("\n" + "=" * 78)
    print(title)
    print(f"Time UTC      : {message_time(msg)}")
    print(f"Source        : {source_string(msg)}")


def print_px4_mode(custom_mode):
    decoded = decode_px4_custom_mode(custom_mode)

    if decoded is None:
        return

    print(f"PX4 mode raw  : {decoded['raw']} ({decoded['hex']})")
    print(
        f"PX4 main mode : {decoded['main_mode']} "
        f"({decoded['main_name']})"
    )
    print(
        f"PX4 submode   : {decoded['sub_mode']} "
        f"({decoded['sub_name']})"
    )


def is_global_frame(frame):
    global_frame_names = [
        "MAV_FRAME_GLOBAL",
        "MAV_FRAME_GLOBAL_RELATIVE_ALT",
        "MAV_FRAME_GLOBAL_INT",
        "MAV_FRAME_GLOBAL_RELATIVE_ALT_INT",
        "MAV_FRAME_GLOBAL_TERRAIN_ALT",
        "MAV_FRAME_GLOBAL_TERRAIN_ALT_INT",
    ]

    global_frames = {
        getattr(mavutil.mavlink, name)
        for name in global_frame_names
        if hasattr(mavutil.mavlink, name)
    }

    return int(frame) in global_frames


def print_command_long(msg):
    command = int(msg.command)

    if not PRINT_ALL_COMMANDS and command not in IMPORTANT_COMMANDS:
        return

    print_header("COMMAND_LONG — direct command", msg)

    print(f"Target        : {msg.target_system}:{msg.target_component}")
    print(f"Command       : {command} ({command_name(command)})")
    print(f"Confirmation  : {msg.confirmation}")

    for index in range(1, 8):
        print(f"Param{index:<2}      : {getattr(msg, f'param{index}')}")

    set_mode_command = getattr(
        mavutil.mavlink,
        "MAV_CMD_DO_SET_MODE",
        176,
    )

    if command == set_mode_command:
        base_mode = int(msg.param1)
        custom_main = int(msg.param2)
        custom_sub = int(msg.param3)

        print("\nDecoded mode request")
        print(f"Base mode     : {base_mode}")
        print(f"Base flags    : {', '.join(decode_mode_flags(base_mode))}")
        print(f"Custom mode   : {custom_main}")
        print(f"Custom submode: {custom_sub}")


def print_command_int(msg):
    command = int(msg.command)
    frame = int(msg.frame)

    if not PRINT_ALL_COMMANDS and command not in IMPORTANT_COMMANDS:
        return

    print_header("COMMAND_INT — direct positional/action command", msg)

    print(f"Target        : {msg.target_system}:{msg.target_component}")
    print(f"Command       : {command} ({command_name(command)})")
    print(f"Frame         : {frame} ({frame_name(frame)})")
    print(f"Current       : {msg.current}")
    print(f"Autocontinue  : {msg.autocontinue}")

    print(f"Param1        : {msg.param1}")
    print(f"Param2        : {msg.param2}")
    print(f"Param3        : {msg.param3}")
    print(f"Param4        : {msg.param4}")

    if is_global_frame(frame):
        latitude = msg.x / 1e7
        longitude = msg.y / 1e7

        print(f"Latitude      : {latitude}")
        print(f"Longitude     : {longitude}")
        print(f"Altitude      : {msg.z}")
    else:
        # COMMAND_INT local coordinates are scaled by 1e4.
        print(f"Local X       : {msg.x / 1e4}")
        print(f"Local Y       : {msg.y / 1e4}")
        print(f"Local Z       : {msg.z}")

    reposition_command = getattr(
        mavutil.mavlink,
        "MAV_CMD_DO_REPOSITION",
        192,
    )

    if command == reposition_command:
        print("\n>>> This is an off-mission GO TO / REPOSITION command.")


def print_set_mode(msg):
    print_header("SET_MODE — mode-change request", msg)

    base_mode = int(msg.base_mode)
    custom_mode = int(msg.custom_mode)

    print(f"Target system : {msg.target_system}")
    print(f"Base mode     : {base_mode}")
    print(f"Base flags    : {', '.join(decode_mode_flags(base_mode))}")
    print(f"Custom mode   : {custom_mode}")

    print_px4_mode(custom_mode)


def print_global_position_target(msg):
    print_header(
        "SET_POSITION_TARGET_GLOBAL_INT — direct global setpoint",
        msg,
    )

    print(f"Target        : {msg.target_system}:{msg.target_component}")
    print(
        f"Frame         : {msg.coordinate_frame} "
        f"({frame_name(msg.coordinate_frame)})"
    )
    print(f"Type mask     : {msg.type_mask} (0x{msg.type_mask:04X})")
    print(f"Latitude      : {msg.lat_int / 1e7}")
    print(f"Longitude     : {msg.lon_int / 1e7}")
    print(f"Altitude      : {msg.alt}")
    print(f"Velocity NED  : vx={msg.vx}, vy={msg.vy}, vz={msg.vz}")
    print(f"Yaw           : {msg.yaw}")
    print(f"Yaw rate      : {msg.yaw_rate}")

    print("\n>>> This is a direct position/offboard setpoint, not a mission item.")


def print_local_position_target(msg):
    print_header(
        "SET_POSITION_TARGET_LOCAL_NED — direct local setpoint",
        msg,
    )

    print(f"Target        : {msg.target_system}:{msg.target_component}")
    print(
        f"Frame         : {msg.coordinate_frame} "
        f"({frame_name(msg.coordinate_frame)})"
    )
    print(f"Type mask     : {msg.type_mask} (0x{msg.type_mask:04X})")
    print(f"Position NED  : x={msg.x}, y={msg.y}, z={msg.z}")
    print(f"Velocity NED  : vx={msg.vx}, vy={msg.vy}, vz={msg.vz}")
    print(f"Yaw           : {msg.yaw}")
    print(f"Yaw rate      : {msg.yaw_rate}")

    print("\n>>> This is a direct local movement setpoint, not a mission item.")


def print_command_ack(msg):
    print_header("COMMAND_ACK — command result", msg)

    command = int(msg.command)
    result = int(msg.result)

    print(f"Command       : {command} ({command_name(command)})")
    print(f"Result        : {result} ({enum_name('MAV_RESULT', result)})")

    if hasattr(msg, "progress"):
        print(f"Progress      : {msg.progress}")

    if hasattr(msg, "result_param2"):
        print(f"Result param2 : {msg.result_param2}")

    if hasattr(msg, "target_system"):
        print(
            f"ACK target    : "
            f"{msg.target_system}:{msg.target_component}"
        )


def extract_non_mission_actions(tlog_path):
    mav = mavutil.mavlink_connection(
        tlog_path,
        robust_parsing=True,
    )

    previous_heartbeat_mode = {}
    previous_extended_state = {}

    counters = {
        "COMMAND_LONG": 0,
        "COMMAND_INT": 0,
        "SET_MODE": 0,
        "SET_POSITION_TARGET_GLOBAL_INT": 0,
        "SET_POSITION_TARGET_LOCAL_NED": 0,
        "COMMAND_ACK": 0,
        "HEARTBEAT_MODE_CHANGE": 0,
        "EXTENDED_SYS_STATE_CHANGE": 0,
    }

    while True:
        msg = mav.recv_match(blocking=False)

        if msg is None:
            break

        msg_type = msg.get_type()

        if msg_type == "BAD_DATA":
            continue

        if msg_type == "COMMAND_LONG":
            command = int(msg.command)

            if PRINT_ALL_COMMANDS or command in IMPORTANT_COMMANDS:
                counters["COMMAND_LONG"] += 1
                print_command_long(msg)

        elif msg_type == "COMMAND_INT":
            command = int(msg.command)

            if PRINT_ALL_COMMANDS or command in IMPORTANT_COMMANDS:
                counters["COMMAND_INT"] += 1
                print_command_int(msg)

        elif msg_type == "SET_MODE":
            counters["SET_MODE"] += 1
            print_set_mode(msg)

        elif msg_type == "SET_POSITION_TARGET_GLOBAL_INT":
            counters["SET_POSITION_TARGET_GLOBAL_INT"] += 1
            print_global_position_target(msg)

        elif msg_type == "SET_POSITION_TARGET_LOCAL_NED":
            counters["SET_POSITION_TARGET_LOCAL_NED"] += 1
            print_local_position_target(msg)

        elif msg_type == "COMMAND_ACK":
            counters["COMMAND_ACK"] += 1
            print_command_ack(msg)

        elif msg_type == "HEARTBEAT":
            # HEARTBEAT reports the resulting/current mode.
            source = source_string(msg)
            current_mode = (
                int(msg.base_mode),
                int(msg.custom_mode),
            )

            if previous_heartbeat_mode.get(source) != current_mode:
                previous_heartbeat_mode[source] = current_mode
                counters["HEARTBEAT_MODE_CHANGE"] += 1

                print_header(
                    "HEARTBEAT — current mode changed or first observed",
                    msg,
                )

                print(f"Vehicle type  : {enum_name('MAV_TYPE', msg.type)}")
                print(
                    f"Autopilot     : "
                    f"{enum_name('MAV_AUTOPILOT', msg.autopilot)}"
                )
                print(f"Base mode     : {msg.base_mode}")
                print(
                    f"Base flags    : "
                    f"{', '.join(decode_mode_flags(msg.base_mode))}"
                )
                print(f"Custom mode   : {msg.custom_mode}")

                print_px4_mode(msg.custom_mode)

        elif msg_type == "EXTENDED_SYS_STATE":
            # This is state telemetry, not the original command.
            source = source_string(msg)
            current_state = (
                int(msg.landed_state),
                int(msg.vtol_state),
            )

            if previous_extended_state.get(source) != current_state:
                previous_extended_state[source] = current_state
                counters["EXTENDED_SYS_STATE_CHANGE"] += 1

                print_header(
                    "EXTENDED_SYS_STATE — vehicle state changed",
                    msg,
                )

                print(
                    f"Landed state  : {msg.landed_state} "
                    f"({enum_name('MAV_LANDED_STATE', msg.landed_state)})"
                )
                print(
                    f"VTOL state    : {msg.vtol_state} "
                    f"({enum_name('MAV_VTOL_STATE', msg.vtol_state)})"
                )

    print("\n" + "#" * 78)
    print("Finished extracting non-mission actions")

    for message_type, count in counters.items():
        print(f"{message_type:<34}: {count}")

    print("#" * 78)


extract_non_mission_actions(TLOG_PATH)


EXTENDED_SYS_STATE — vehicle state changed
Time UTC      : 2026-07-11T21:32:18.747000+00:00
Source        : 1:1
Landed state  : 1 (MAV_LANDED_STATE_ON_GROUND)
VTOL state    : 0 (MAV_VTOL_STATE_UNDEFINED)

COMMAND_ACK — command result
Time UTC      : 2026-07-11T21:32:19.675000+00:00
Source        : 1:1
Command       : 512 (MAV_CMD_REQUEST_MESSAGE)
Result        : 0 (MAV_RESULT_ACCEPTED)
Progress      : 0
Result param2 : 0
ACK target    : 255:190

HEARTBEAT — current mode changed or first observed
Time UTC      : 2026-07-11T21:32:19.675000+00:00
Source        : 255:190
Vehicle type  : MAV_TYPE_GCS
Autopilot     : MAV_AUTOPILOT_INVALID
Base mode     : 192
Base flags    : MAV_MODE_FLAG_MANUAL_INPUT_ENABLED, MAV_MODE_FLAG_SAFETY_ARMED
Custom mode   : 0
PX4 mode raw  : 0 (0x00000000)
PX4 main mode : 0 (UNKNOWN_PX4_CUSTOM_MAIN_MODE_0)
PX4 submode   : 0 (0)

COMMAND_ACK — command result
Time UTC      : 2026-07-11T21:32:19.853000+00:00
Source        : 1:1
Command       : 512 (MAV_CMD_REQUEST_M

### get the land and RTL without mission

In [ ]:
from pymavlink import mavutil
from copy import deepcopy
import json, math

TLOG = "two_land_and_go_to_waypoint.tlog"
TRANSITION_OUT = "set_mode_transition.jsonl"
SEQUENCE_OUT = "set_mode_sequences.jsonl"

TELEM_GROUPS = {
    "GLOBAL_POSITION_INT": ["lat","lon","alt","relative_alt","vx","vy","vz","hdg"],
    "ATTITUDE": ["roll","pitch","yaw"],
    "VFR_HUD": ["groundspeed","heading","throttle","alt","climb"],
    "SYS_STATUS": ["battery_remaining","voltage_battery","load"],
    "GPS_RAW_INT": ["fix_type"],
}

# (PX4 main mode, submode): (RAG command, mode name)
MODES = {
    (4, 5): (20, "PX4_AUTO_RTL"),
    (4, 6): (21, "PX4_AUTO_LAND"),
}

# Commands that end the current sequence; 512/521 are intentionally omitted.
FLIGHT_COMMANDS = {16, 20, 21, 22, 115, 176, 178, 192, 300, 400}


def clean(v):
    return None if isinstance(v, float) and not math.isfinite(v) else v


def selected_fields(msg, names):
    d = msg.to_dict()
    return {k: clean(d[k]) for k in names if k in d}


def make_request(msg, command, mode_name, main_mode, sub_mode):
    return {
        "type": "SET_MODE",
        "source_msg": "SET_MODE",
        "command": command,                 # RAG alias: RTL=20, LAND=21
        "behavior_command": command,
        "behavior_name": (
            "MAV_CMD_NAV_RETURN_TO_LAUNCH"
            if command == 20 else "MAV_CMD_NAV_LAND"
        ),
        "mode_name": mode_name,
        "target_system": int(msg.target_system),
        "base_mode": int(msg.base_mode),
        "custom_mode": int(msg.custom_mode),
        "main_mode": main_mode,
        "sub_mode": sub_mode,
        **{f"param{i}": 0.0 for i in range(1, 8)},
    }


# ------------------------------------------------------------------
# Parse TLOG
# ------------------------------------------------------------------
log = mavutil.mavlink_connection(TLOG, robust_parsing=True)

events, target_indexes = [], []
last_hb = None
latest_telem = {name: None for name in TELEM_GROUPS}

while True:
    msg = log.recv_match(blocking=False)
    if msg is None:
        break

    msg_type = msg.get_type()
    if msg_type == "BAD_DATA":
        continue

    ts = float(getattr(msg, "_timestamp", 0.0))

    # Keep only vehicle heartbeat type=2.
    if msg_type == "HEARTBEAT" and int(msg.type) == 2:
        last_hb = {
            "base_mode": int(msg.base_mode),
            "custom_mode": int(msg.custom_mode),
            "system_status": int(msg.system_status),
            "autopilot": int(msg.autopilot),
            "mavlink_version": int(msg.mavlink_version),
        }
        events.append({"kind": "hb", "ts": ts, "data": deepcopy(last_hb)})

    elif msg_type in TELEM_GROUPS:
        data = selected_fields(msg, TELEM_GROUPS[msg_type])
        latest_telem[msg_type] = data
        events.append({
            "kind": "telem", "ts": ts,
            "name": msg_type, "data": data
        })

    elif msg_type == "SET_MODE":
        custom_mode = int(msg.custom_mode)
        main_mode = (custom_mode >> 16) & 0xFF
        sub_mode = (custom_mode >> 24) & 0xFF
        mode = MODES.get((main_mode, sub_mode))

        event = {"kind": "mode", "ts": ts}

        if mode:
            command, mode_name = mode
            event.update({
                "request": make_request(
                    msg, command, mode_name, main_mode, sub_mode
                ),
                "prev_hb": deepcopy(last_hb),
                "prev_telem": deepcopy(latest_telem),
            })
            target_indexes.append(len(events))

        events.append(event)

    elif msg_type in ("COMMAND_LONG", "COMMAND_INT"):
        command = int(getattr(msg, "command", -1))
        if command in FLIGHT_COMMANDS:
            events.append({"kind": "command", "ts": ts, "command": command})


# ------------------------------------------------------------------
# Build datasets
# ------------------------------------------------------------------
transitions, sequences = [], []

for i in target_indexes:
    event = events[i]
    request = event["request"]

    # Stop before the next SET_MODE or traceable flight command.
    boundary = next(
        (j for j in range(i + 1, len(events))
         if events[j]["kind"] in ("mode", "command")),
        None,
    )
    end = boundary if boundary is not None else len(events)
    future = events[i + 1:end]

    first_hb = next((x for x in future if x["kind"] == "hb"), None)
    confirmation = next(
        (x for x in future
         if x["kind"] == "hb"
         and x["data"]["custom_mode"] == request["custom_mode"]),
        None,
    )

    # Transition: first message from each preferred telemetry group.
    next_telem = {}
    for x in future:
        if x["kind"] == "telem" and x["name"] not in next_telem:
            next_telem[x["name"]] = x["data"]
            if len(next_telem) == len(TELEM_GROUPS):
                break

    transitions.append({
        "t_cmd": event["ts"],
        "Prev_HB": (
            {"type": 2, **event["prev_hb"]}
            if event["prev_hb"] else None
        ),
        "Prev_Telemetry": event["prev_telem"],
        "Command": request,
        "Command_name": request["mode_name"],
        "t_ack": None,
        "Command_ACK": None,
        "t_confirmation": confirmation["ts"] if confirmation else None,
        "Mode_Confirmation": (
            {"type": "HEARTBEAT", **confirmation["data"]}
            if confirmation else None
        ),
        "HB_NEXT": (
            {"type": 2, **first_hb["data"]}
            if first_hb else None
        ),
        "NEXT_Telemetry": next_telem,
    })

    # Sequence: until boundary; if no boundary, stop after 10 telemetry.
    # followups, telem_count = [], 0

    # for x in future:
    #     if x["kind"] == "hb":
    #         followups.append({
    #             "type": "HEARTBEAT",
    #             "_ts": x["ts"],
    #             **x["data"],
    #         })

    #     elif x["kind"] == "telem":
    #         followups.append({
    #             "type": x["name"],
    #             "_ts": x["ts"],
    #             **x["data"],
    #         })
    #         telem_count += 1

    #         if boundary is None and telem_count >= 10:
    #             break

    # Sequence: maximum 10 messages of each type
    followups = []
    counts = {"HEARTBEAT": 0, **{name: 0 for name in TELEM_GROUPS}}
    mode_confirmed = False

    for x in future:

        # Stop when heartbeat changes away from LAND/RTL
        if x["kind"] == "hb":
            current_mode = x["data"]["custom_mode"]

            if current_mode == request["custom_mode"]:
                mode_confirmed = True

            elif mode_confirmed:
                break

        name = "HEARTBEAT" if x["kind"] == "hb" else x.get("name")

        if name not in counts or counts[name] >= 10:
            continue

        followups.append({
            "type": name,
            "_ts": x["ts"],
            **x["data"],
        })

        counts[name] += 1

        if all(count == 10 for count in counts.values()):
            break

    sequences.append({
        "t_request": event["ts"],
        "t_ack": None,
        "dt_ack_s": None,
        "t_confirmation": confirmation["ts"] if confirmation else None,
        "dt_confirmation_s": (
            round(confirmation["ts"] - event["ts"], 3)
            if confirmation else None
        ),
        "context_prev_heartbeat": (
            {"type": "HEARTBEAT", **event["prev_hb"]}
            if event["prev_hb"] else None
        ),
        "request": request,
        "ack": None,
        "mode_confirmation": (
            {"type": "HEARTBEAT", **confirmation["data"]}
            if confirmation else None
        ),
        "followups": followups,
    })


def save_jsonl(path, rows):
    with open(path, "w") as f:
        for row in rows:
            f.write(json.dumps(row) + "\n")


save_jsonl(TRANSITION_OUT, transitions)
save_jsonl(SEQUENCE_OUT, sequences)

print("Transitions:", len(transitions), "->", TRANSITION_OUT)
print("Sequences:", len(sequences), "->", SEQUENCE_OUT)
print([row["Command_name"] for row in transitions])

Transitions: 2 -> set_mode_transition.jsonl
Sequences: 2 -> set_mode_sequences.jsonl
['PX4_AUTO_LAND', 'PX4_AUTO_LAND']


In [5]:
ls

 extract_tlog_and_Plan.ipynb   mission_alt_yaw_speed.plan
 has_landing.tlog              mission_custom_yaw_speed.plan
 has_ROI.tlog                 'two_land_and_go _to_waypoint.tlog'
 mission4_all_combo.plan


In [2]:
import json
from pprint import pprint

try:
    from pymavlink.dialects.v20 import common as mavlink2
    MAV_CMD_ENUM = mavlink2.enums.get("MAV_CMD", {})
    MAV_FRAME_ENUM = mavlink2.enums.get("MAV_FRAME", {})
except Exception:
    MAV_CMD_ENUM = {}
    MAV_FRAME_ENUM = {}


def mav_cmd_name(command):
    try:
        command = int(command)
        return MAV_CMD_ENUM[command].name if command in MAV_CMD_ENUM else f"UNKNOWN_MAV_CMD_{command}"
    except Exception:
        return "UNKNOWN_MAV_CMD"


def mav_frame_name(frame):
    try:
        frame = int(frame)
        return MAV_FRAME_ENUM[frame].name if frame in MAV_FRAME_ENUM else f"UNKNOWN_FRAME_{frame}"
    except Exception:
        return "UNKNOWN_FRAME"


def normalize_params(params):
    params = list(params or [])
    while len(params) < 7:
        params.append(None)
    return params[:7]


def extract_mission_items_from_plan(plan_file):
    with open(plan_file, "r") as f:
        plan = json.load(f)

    mission = plan.get("mission", {})
    items = mission.get("items", [])

    extracted = []

    def walk(items, parent_type=None):
        for item in items:
            item_type = item.get("type", "Unknown")

            if item_type == "SimpleItem":
                command = item.get("command")
                frame = item.get("frame")
                params = normalize_params(item.get("params"))

                p1, p2, p3, p4, lat, lon, alt = params

                extracted.append({
                    "item_type": "SimpleItem",
                    "doJumpId": item.get("doJumpId"),
                    "command": command,
                    "command_name": mav_cmd_name(command),
                    "frame": frame,
                    "frame_name": mav_frame_name(frame),
                    "autoContinue": item.get("autoContinue"),
                    "param1": p1,
                    "param2": p2,
                    "param3": p3,
                    "param4": p4,
                    "latitude": lat,
                    "longitude": lon,
                    "altitude": alt,
                    "parent_type": parent_type,
                    "raw": item,
                })

            elif item_type == "ComplexItem":
                complex_type = item.get("complexItemType", "UnknownComplexItem")

                extracted.append({
                    "item_type": "ComplexItem",
                    "complexItemType": complex_type,
                    "parent_type": parent_type,
                    "raw": item,
                })

                nested_items = item.get("Items") or item.get("items") or []
                if nested_items:
                    walk(nested_items, parent_type=complex_type)

            elif item_type == "MissionSettings":
                extracted.append({
                    "item_type": "MissionSettings",
                    "parent_type": parent_type,
                    "raw": item,
                })

            else:
                extracted.append({
                    "item_type": item_type,
                    "parent_type": parent_type,
                    "raw": item,
                })

    walk(items)
    return extracted


def print_mission_items(mission_items):
    print("\n========== EXTRACTED MISSION ITEMS ==========\n")

    for i, item in enumerate(mission_items, start=1):
        print(f"Item {i}")
        print(f"  item_type     : {item.get('item_type')}")

        if item.get("item_type") == "SimpleItem":
            print(f"  doJumpId      : {item.get('doJumpId')}")
            print(f"  command       : {item.get('command')} ({item.get('command_name')})")
            print(f"  frame         : {item.get('frame')} ({item.get('frame_name')})")
            print(f"  autoContinue  : {item.get('autoContinue')}")
            print(f"  param1        : {item.get('param1')}")
            print(f"  param2        : {item.get('param2')}")
            print(f"  param3        : {item.get('param3')}")
            print(f"  param4        : {item.get('param4')}")
            print(f"  latitude      : {item.get('latitude')}")
            print(f"  longitude     : {item.get('longitude')}")
            print(f"  altitude      : {item.get('altitude')}")

        elif item.get("item_type") == "ComplexItem":
            print(f"  complexType   : {item.get('complexItemType')}")

        elif item.get("item_type") == "MissionSettings":
            print("  Mission settings item found.")

        else:
            print("  Unknown item type found.")

        if item.get("parent_type"):
            print(f"  parent_type   : {item.get('parent_type')}")

        print("-" * 60)


plan_file = "two_land_and_go_to_waypoint.plan"   # change path if needed

mission_items = extract_mission_items_from_plan(plan_file)
print_mission_items(mission_items)

FileNotFoundError: [Errno 2] No such file or directory: 'two_land_and_go_to_waypoint.plan'

In [3]:
pwd

'/home/vboxuser/Desktop/UAV_tools/Codes/tests/tlog_processing'